<a href="https://colab.research.google.com/github/chrliem/ecommerce-customer-retention/blob/main/E_Commerce_Customer_Retention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **E-Commerce Customer Retention**

Christian Darma Setiawan

**Problem Statement:**
In e-commerce, the cost of customer acquisition is usually higher than the cost of retention. A high churn rate acts like a situation where marketing efforts are wasted as customers leave the platform. Identifying at-risk customers before they leave the platform is critical for maintaining a profitable dan sustainable business

**Objective:**
To analyze key behavioral patterns that drive customer attrition and build a predictive model that allows the business to prepare data-driven retention strategies.

Dataset [E-Commerce Customer Churn](https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction/data)

In [84]:
import pandas as pd
import requests
import time
import os
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots

In [85]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Data Ovewview**

In [86]:
base_path = '/content/drive/MyDrive/'
file_name = 'ecommerce-dataset.xlsx'
full_path = os.path.join(base_path, file_name)

if os.path.exists(full_path):
    df = pd.read_excel(full_path, sheet_name='E Comm')

    date_cols = [col for col in df.columns if 'date' in col.lower()]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    print(f"{'Churn Data'.ljust(18)} | Rows: {len(df):>8,} | Cols: {len(df.columns)}")
    print("\n" + "="*50 + "\n")

    display(df.head())
else:
    print(f"Error: {file_name} tidak ditemukan di {full_path}!")


Churn Data         | Rows:    5,630 | Cols: 20




,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [87]:
df.shape

(5630, 20)

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

## **Data Quality Assesment**

In [89]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100

null_summary = pd.DataFrame({
    'Total Null': null_counts,
    'Percentage (%)': null_pct.round(2)
}).filter(items = null_counts[null_counts > 0].index, axis=0).sort_values('Total Null', ascending=False)

if not null_summary.empty:
    display(null_summary)
else:
    print("No missing values found.")

duplicate_count = df.duplicated().sum()
print(f"\nTotal Duplicate Rows: {duplicate_count}")

unique_id = df['CustomerID'].nunique()
is_unique = unique_id == len(df)
print(f"Unique Customers: {unique_id} | Is every row a unique customer? {is_unique}")

,Total Null,Percentage (%)
DaySinceLastOrder,307,5.45
OrderAmountHikeFromlastYear,265,4.71
Tenure,264,4.69
OrderCount,258,4.58
CouponUsed,256,4.55
HourSpendOnApp,255,4.53
WarehouseToHome,251,4.46



Total Duplicate Rows: 0
Unique Customers: 5630 | Is every row a unique customer? True


In [90]:
num_features = ['Tenure', 'WarehouseToHome', 'DaySinceLastOrder', 'OrderAmountHikeFromlastYear', 'CashbackAmount']

for col in num_features:
    fig = px.box(df,
                 y=col,
                 x='Churn',
                 color='Churn',
                 points="all",
                 notched=True,
                 title=f"Outlier & Distribution: {col}",
                 color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
                 template='plotly_white')

    fig.update_layout(
        xaxis_title="Status Churn (0: Stay, 1: Churn)",
        yaxis_title=f"Nilai {col}",
        showlegend=False
    )
    fig.show()

In [91]:
churn_dist = df['Churn'].map({0: 'Stay', 1: 'Churn'}).value_counts()
churn_pct = df['Churn'].map({0: 'Stay', 1: 'Churn'}).value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count': churn_dist,
    'Percentage (%)': churn_pct.round(2)
})
display(target_summary)

fig = px.bar(target_summary, x=target_summary.index, y='Count',
             text='Percentage (%)', title='Distribution of Churn',
             color=target_summary.index,
             color_discrete_map={'Stay': '#2ecc71', 'Churn': '#e74c3c'},
             labels={'index': 'Customer Status'})

fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()

,Count,Percentage (%)
Churn,,
Stay,4682,83.16
Churn,948,16.84


In [92]:
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    unique_vals = df[col].unique()
    num_unique = df[col].nunique()
    print(f"\nColumn: {col} ({num_unique} unique values)")
    print(f"Values: {unique_vals}")


Column: PreferredLoginDevice (3 unique values)
Values: ['Mobile Phone' 'Phone' 'Computer']

Column: PreferredPaymentMode (7 unique values)
Values: ['Debit Card' 'UPI' 'CC' 'Cash on Delivery' 'E wallet' 'COD' 'Credit Card']

Column: Gender (2 unique values)
Values: ['Female' 'Male']

Column: PreferedOrderCat (6 unique values)
Values: ['Laptop & Accessory' 'Mobile' 'Mobile Phone' 'Others' 'Fashion' 'Grocery']

Column: MaritalStatus (3 unique values)
Values: ['Single' 'Divorced' 'Married']


In [93]:
import math

num_cols = df.select_dtypes(exclude=['object']).columns.drop(['CustomerID', 'Churn'])

display(df[num_cols].describe().T)


,count,mean,std,min,25%,50%,75%,max
Tenure,5366.0,10.189899,8.557241,0.0,2.00,9.00,16.0000,61.00
CityTier,5630.0,1.654707,0.915389,1.0,1.00,1.00,3.0000,3.00
WarehouseToHome,5379.0,15.639896,8.531475,5.0,9.00,14.00,20.0000,127.00
HourSpendOnApp,5375.0,2.931535,0.721926,0.0,2.00,3.00,3.0000,5.00
NumberOfDeviceRegistered,5630.0,3.688988,1.023999,1.0,3.00,4.00,4.0000,6.00
SatisfactionScore,5630.0,3.066785,1.380194,1.0,2.00,3.00,4.0000,5.00
NumberOfAddress,5630.0,4.214032,2.583586,1.0,2.00,3.00,6.0000,22.00
Complain,5630.0,0.284902,0.451408,0.0,0.00,0.00,1.0000,1.00
OrderAmountHikeFromlastYear,5365.0,15.707922,3.675485,11.0,13.00,15.00,18.0000,26.00
CouponUsed,5374.0,1.751023,1.894621,0.0,1.00,1.00,2.0000,16.00


In [94]:
for col in num_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"Column {col} has {neg_count} negative values!")
    else:
        print(f"Column {col}: No negative values.")

Column Tenure: No negative values.
Column CityTier: No negative values.
Column WarehouseToHome: No negative values.
Column HourSpendOnApp: No negative values.
Column NumberOfDeviceRegistered: No negative values.
Column SatisfactionScore: No negative values.
Column NumberOfAddress: No negative values.
Column Complain: No negative values.
Column OrderAmountHikeFromlastYear: No negative values.
Column CouponUsed: No negative values.
Column OrderCount: No negative values.
Column DaySinceLastOrder: No negative values.
Column CashbackAmount: No negative values.


## **Data Cleaning**

In [95]:
def clean_ecommerce_data(df_input):
    df_clean = df_input.copy()

    df_clean['PreferredLoginDevice'] = df_clean['PreferredLoginDevice'].replace('Phone', 'Mobile Phone')

    df_clean['PreferredPaymentMode'] = df_clean['PreferredPaymentMode'].replace({
        'CC': 'Credit Card',
        'COD': 'Cash on Delivery'
    })

    df_clean['PreferedOrderCat'] = df_clean['PreferedOrderCat'].replace('Mobile', 'Mobile Phone')

    num_cols_with_null = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp',
                          'OrderAmountHikeFromlastYear', 'CouponUsed',
                          'OrderCount', 'DaySinceLastOrder']

    for col in num_cols_with_null:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    int_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'OrderCount', 'DaySinceLastOrder']
    df_clean[int_cols] = df_clean[int_cols].astype(int)

    return df_clean

df_cleaned = clean_ecommerce_data(df)

print(f"Missing Values remaining: {df_cleaned.isnull().sum().sum()}")
display(df_cleaned.describe())

Missing Values remaining: 0


,CustomerID,Churn,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
count,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000
mean,52815.500000,0.168384,10.134103,1.654707,15.566785,2.934636,3.688988,3.066785,4.214032,0.284902,15.674600,1.716874,2.961812,4.459325,177.223030
std,1625.385339,0.374240,8.357951,0.915389,8.345961,0.705528,1.023999,1.380194,2.583586,0.451408,3.591058,1.857640,2.879248,3.570626,49.207036
min,50001.000000,0.000000,0.000000,1.000000,5.000000,0.000000,1.000000,1.000000,1.000000,0.000000,11.000000,0.000000,1.000000,0.000000,0.000000
25%,51408.250000,0.000000,3.000000,1.000000,9.000000,2.000000,3.000000,2.000000,2.000000,0.000000,13.000000,1.000000,1.000000,2.000000,145.770000
50%,52815.500000,0.000000,9.000000,1.000000,14.000000,3.000000,4.000000,3.000000,3.000000,0.000000,15.000000,1.000000,2.000000,3.000000,163.280000
75%,54222.750000,0.000000,15.000000,3.000000,20.000000,3.000000,4.000000,4.000000,6.000000,1.000000,18.000000,2.000000,3.000000,7.000000,196.392500
max,55630.000000,1.000000,61.000000,3.000000,127.000000,5.000000,6.000000,5.000000,22.000000,1.000000,26.000000,16.000000,16.000000,46.000000,324.990000


In [96]:
cols_to_compare = ['PreferredLoginDevice', 'PreferredPaymentMode', 'PreferedOrderCat']

for col in cols_to_compare:
    before = df[col].value_counts().sort_index()
    after = df_cleaned[col].value_counts().sort_index()

    comparison = pd.DataFrame({'Before Cleaning': before, 'After Cleaning': after}).fillna(0).astype(int)
    print(f"\nFeature: {col}")
    display(comparison)


Feature: PreferredLoginDevice


,Before Cleaning,After Cleaning
PreferredLoginDevice,,
Computer,1634,1634
Mobile Phone,2765,3996
Phone,1231,0



Feature: PreferredPaymentMode


,Before Cleaning,After Cleaning
PreferredPaymentMode,,
CC,273,0
COD,365,0
Cash on Delivery,149,514
Credit Card,1501,1774
Debit Card,2314,2314
E wallet,614,614
UPI,414,414



Feature: PreferedOrderCat


,Before Cleaning,After Cleaning
PreferedOrderCat,,
Fashion,826,826
Grocery,410,410
Laptop & Accessory,2050,2050
Mobile,809,0
Mobile Phone,1271,2080
Others,264,264


In [97]:
cat_cols = df_cleaned.select_dtypes(include=['object']).columns

for col in cat_cols:
    unique_vals = df_cleaned[col].unique()
    num_unique = df_cleaned[col].nunique()
    print(f"\nColumn: {col} ({num_unique} unique values)")
    print(f"Values: {unique_vals}")


Column: PreferredLoginDevice (2 unique values)
Values: ['Mobile Phone' 'Computer']

Column: PreferredPaymentMode (5 unique values)
Values: ['Debit Card' 'UPI' 'Credit Card' 'Cash on Delivery' 'E wallet']

Column: Gender (2 unique values)
Values: ['Female' 'Male']

Column: PreferedOrderCat (5 unique values)
Values: ['Laptop & Accessory' 'Mobile Phone' 'Others' 'Fashion' 'Grocery']

Column: MaritalStatus (3 unique values)
Values: ['Single' 'Divorced' 'Married']


In [98]:
null_before = df.isnull().sum()
null_after = df_cleaned.isnull().sum()

missing_compare = pd.DataFrame({
    'Before (Null)': null_before,
    'After (Null)': null_after
})

display(missing_compare[missing_compare['Before (Null)'] > 0])

,Before (Null),After (Null)
Tenure,264,0
WarehouseToHome,251,0
HourSpendOnApp,255,0
OrderAmountHikeFromlastYear,265,0
CouponUsed,256,0
OrderCount,258,0
DaySinceLastOrder,307,0


In [99]:
def plot_before_after(col):
    counts_before = df[col].value_counts().sort_index()
    counts_after = df_cleaned[col].value_counts().sort_index()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f'Before Cleaning: {col}', f'After Cleaning: {col}'),
        horizontal_spacing=0.1
    )

    fig.add_trace(
        go.Bar(
            x=counts_before.index,
            y=counts_before.values,
            text=counts_before.values,
            textposition='auto',
            marker_color='#AED6F1',
            name='Before'
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=counts_after.index,
            y=counts_after.values,
            text=counts_after.values,
            textposition='auto',
            marker_color='#16A085',
            name='After'
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=500,
        showlegend=False,
        template='plotly_white'
    )

    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)

    fig.show()

In [100]:
plot_before_after('PreferedOrderCat')

In [101]:
plot_before_after('PreferredPaymentMode')

In [102]:
extreme_cols = ['WarehouseToHome', 'DaySinceLastOrder', 'CouponUsed', 'OrderCount']

for col in extreme_cols:
    upper_limit = df_cleaned[col].quantile(0.99)
    df_cleaned[col] = df_cleaned[col].clip(upper=upper_limit)

df_cleaned.describe()

,CustomerID,Churn,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
count,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000,5630.000000
mean,52815.500000,0.168384,10.134103,1.654707,15.525222,2.934636,3.688988,3.066785,4.214032,0.284902,15.674600,1.691652,2.947780,4.434636,177.223030
std,1625.385339,0.374240,8.357951,0.915389,8.065594,0.705528,1.023999,1.380194,2.583586,0.451408,3.591058,1.728751,2.820963,3.455667,49.207036
min,50001.000000,0.000000,0.000000,1.000000,5.000000,0.000000,1.000000,1.000000,1.000000,0.000000,11.000000,0.000000,1.000000,0.000000,0.000000
25%,51408.250000,0.000000,3.000000,1.000000,9.000000,2.000000,3.000000,2.000000,2.000000,0.000000,13.000000,1.000000,1.000000,2.000000,145.770000
50%,52815.500000,0.000000,9.000000,1.000000,14.000000,3.000000,4.000000,3.000000,3.000000,0.000000,15.000000,1.000000,2.000000,3.000000,163.280000
75%,54222.750000,0.000000,15.000000,3.000000,20.000000,3.000000,4.000000,4.000000,6.000000,1.000000,18.000000,2.000000,3.000000,7.000000,196.392500
max,55630.000000,1.000000,61.000000,3.000000,35.000000,5.000000,6.000000,5.000000,22.000000,1.000000,26.000000,9.000000,14.000000,15.000000,324.990000


## **EDA**

### Does the distance between the warehouse and customer significantly impact the likelihood of churn?



In [103]:
fig = px.histogram(df_cleaned, x="WarehouseToHome", color="Churn",
                   marginal="box",
                   barmode="overlay",
                   histnorm='probability density',
                   title="Impact of Warehouse Distance on Customer Churn",
                   labels={'WarehouseToHome': 'Distance from Warehouse to Home', 'Churn': 'Churn Status'},
                   color_discrete_map={0: '#3498db', 1: '#e74c3c'},
                   template="plotly_white",
                   width=1000
                  )
fig.for_each_trace(lambda t: t.update(name = "Churn" if t.name == "1" else "Stay"))
fig.update_layout(bargap=0.1,
                  xaxis_title="Distance (km)",
                  legend_title="Status")
fig.show()

From the visualization, loyal customers are most dense at 9 km radius and the peak of churned customers are at 14 km. Although both groups share the same median at 14 km, there is a divergence at Q3 where 75% of staying customers reside within 19 km, compared to 22 km for thos who churn. This suggest that 19-22 km range can be a threshold where distance likely drives attrition and confirming that customers living beyond 19 km have a higher probability to leave the platform.

### Are customers who file complaints actualy more likely to churn or is customers type who simply stop ordering a bigger threat?

In [104]:
complain_data = df_cleaned.groupby(['Complain', 'Churn']).size().reset_index(name='count')
complain_data['percentage'] = complain_data.groupby('Complain')['count'].transform(lambda x: (x / x.sum()) * 100)

complain_data['Complain'] = complain_data['Complain'].map({0: 'No Complaint', 1: 'Complaint Filed'})
complain_data['Churn'] = complain_data['Churn'].map({0: 'Stay', 1: 'Churn'})

fig = px.bar(complain_data, x="Complain", y="percentage", color="Churn",
             title="Churn Rate Analysis: The Impact of Customer Complaints",
             labels={'percentage': 'Percentage (%)'},
             text=complain_data['percentage'].apply(lambda x: f'{x:.1f}%'),
             color_discrete_map={'Stay': '#AED6F1', 'Churn': '#E74C3C'},
             template="plotly_white",
             barmode="stack",
             width=800
             )

fig.update_xaxes(title_text=None)
fig.update_traces(textposition='inside')
fig.show()

The result reveals a contrast between customer complaints and the decision to churn. Customers who complained exhibit a churn rate 31.7%, which is nearly triple the rate of customer that never complained 10.9%.
This silent churn remains a significant threat that may undetected.

### Is high cashback effective in buying long-term loyalty? (high cashback = above median 163.28)

In [105]:
import plotly.express as px

df_temp = df_cleaned.copy()

median_cb = df_temp['CashbackAmount'].median()
df_temp['Cashback_Tier'] = df_temp['CashbackAmount'].apply(
    lambda x: 'High Cashback' if x > median_cb else 'Standard Cashback'
)

trend_df = df_temp.groupby(['Tenure', 'Cashback_Tier'])['Churn'].mean().reset_index()
trend_df['Churn Rate (%)'] = trend_df['Churn'] * 100

trend_df['Smoothed Churn (%)'] = trend_df.groupby('Cashback_Tier')['Churn Rate (%)'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

fig = px.line(trend_df,
              x="Tenure",
              y="Smoothed Churn (%)",
              color="Cashback_Tier",
              title="High Cashback vs Loyalty",
              labels={'Tenure': 'Tenure (Months)', 'Smoothed Churn (%)': 'Churn Risk (%)'},
              color_discrete_map={'High Cashback': '#2ecc71', 'Standard Cashback': '#e74c3c'},
              template="plotly_white",
              width=1000)

fig.update_traces(line=dict(width=3))
fig.update_layout(legend_title_text='Cashback Received')
fig.show()

The result above reveals that high cashback segment shows a significantly higher initial churn risk (67.7%) in the first month compared to standard cashback group (51.5%). This confirms the financial rewards primarily attract promo hunters where they use the big cashback during initial usage and exit immediately.


Once the customer survive the first 6 months, the impact of cashback amount diminishes as the primary driver shift to organic.

The most significant milestone occurs at the 24 month where churn risk for both groups hits 0% which might indicate absolute retention This users who remain past the 2 years transform into a permanent user and tenure effectively eliminates the risk of churn.

### Feature Correlation

In [106]:
corr_with_churn = df_cleaned.select_dtypes(include='number').corr()[['Churn']].sort_values(by='Churn', ascending=False)

corr_with_churn = corr_with_churn.drop('Churn')

fig = px.bar(corr_with_churn,
             x=corr_with_churn.index,
             y='Churn',
             text_auto='.2f',
             title="Feature Importance: What Drives Customer Churn?",
             labels={'index': 'Features', 'Churn': 'Correlation Coefficient'},
             color='Churn',
             color_continuous_scale='RdBu_r',
             template="plotly_white")

fig.update_layout(coloraxis_showscale=False)
fig.show()

The feature correlation provides a view of which drivers behind customer churn. From the result, Tenure is the strongest individual predictor with a negative correlation of -0.34, which confirms that the longer a customer stays, the less likely they are to churn. On the other hand, Complain exhibits the strongest positive correlation with churn at 0.25 which confirms it as a primary red flag for immediate customer exit. Other negative correlations like DaySinceLastOrder (-0.16) and CashbackAmount (-0.15) suggests that recent engagement and financial incentives play a meaninfgul roles in retention also.

### Does the way a customer accesses the platform affect their long term retention?

In [107]:
device_df = df_cleaned.copy()
device_df['Customer_Status'] = device_df['Churn'].map({0: 'Stay', 1: 'Churn'})

device_churn = device_df.groupby(['PreferredLoginDevice', 'Customer_Status']).size().reset_index(name='count')
device_churn['percentage'] = device_churn.groupby('PreferredLoginDevice')['count'].transform(lambda x: (x / x.sum()) * 100)

fig = px.bar(device_churn,
             x="PreferredLoginDevice",
             y="percentage",
             color="Customer_Status",
             barmode="group",
             text=device_churn['percentage'].apply(lambda x: f'{x:.1f}%'),
             title="Churn Risk Across Different Login Devices",
             labels={'percentage': 'Percentage (%)', 'PreferredLoginDevice': 'Device Type'},
             color_discrete_map={'Stay': '#3498db', 'Churn': '#e74c3c'},
             template="plotly_white")

fig.update_traces(textposition='outside')
fig.show()

The comparison of churn rates by login device shows that customers who using a computer exhibit a higher churn rate of 19.8% compared to mobile phone users at 15.6%. This suggests  mobile users are slightly more loyal and the desktop experience might be not as easier rather than mboile which lead to higher churn on computer users.

### Are users with high coupon usage tend to churn compared to organic customers once the discounts stop?

In [108]:
df_plot = df_cleaned.copy()
df_plot['Customer_Status'] = df_plot['Churn'].map({0: 'Stay', 1: 'Churn'})

fig = px.box(df_plot,
             x="Customer_Status",
             y="CouponUsed",
             color="Customer_Status",
             points="outliers",
             title="Coupon Dependency: Are Churners Promo-Driven?",
             labels={'CouponUsed': 'Total Coupons Used', 'Customer_Status': 'Status'},
             color_discrete_map={'Stay': '#2ecc71', 'Churn': '#e74c3c'},
             template="plotly_white",
             width=800)

fig.update_layout(showlegend=False)
fig.show()

The result shows a similar pattern between churned and stayed customers where it has the same median usage and interquartile. This indicates that the total quantity of coupons used is not a standalone predictor of churn, but more likely in how dependent a customer becomes on these incentives to maintain activity.

### Does a low satisfaction score lead to churn or is there something before a customer decides to leave?

In [109]:
sat_df = df_cleaned.copy()
sat_df['Customer_Status'] = sat_df['Churn'].map({0: 'Stay', 1: 'Churn'})

sat_churn = sat_df.groupby(['SatisfactionScore', 'Customer_Status']).size().reset_index(name='count')
sat_churn['percentage'] = sat_churn.groupby('SatisfactionScore')['count'].transform(lambda x: (x / x.sum()) * 100)

fig = px.bar(sat_churn,
             x="SatisfactionScore",
             y="percentage",
             color="Customer_Status",
             barmode="group",
             text=sat_churn['percentage'].apply(lambda x: f'{x:.1f}%'),
             title="The Satisfaction vs. Churn Risk",
             labels={'percentage': 'Percentage (%)', 'SatisfactionScore': 'Satisfaction Rating (1-5)'},
             color_discrete_map={'Stay': '#3498db', 'Churn': '#e74c3c'},
             template="plotly_white")

fig.update_xaxes(dtick=1)
fig.update_traces(textposition='outside')
fig.show()

The churn probability here shows something that unusual where higher satisfaction lead to a higher likelihood of churn. While customers with a score of 1 or 2 shows a relatively low churn probability, those who give higher rating exhibit higher risk of attrition. It means that a high score cannot be an indicator of loyalty alone,

### Is there a significant difference in churn rates between customers who prefere specific categoris?

In [110]:
cat_df = df_cleaned.copy()
cat_df['Customer_Status'] = cat_df['Churn'].map({0: 'Stay', 1: 'Churn'})

cat_churn = cat_df.groupby(['PreferedOrderCat', 'Customer_Status']).size().reset_index(name='count')
cat_churn['percentage'] = cat_churn.groupby('PreferedOrderCat')['count'].transform(lambda x: (x / x.sum()) * 100)

churn_rates = cat_churn[cat_churn['Customer_Status'] == 'Churn'].sort_values(by='percentage', ascending=True)
category_order = churn_rates['PreferedOrderCat'].tolist()

fig = px.bar(cat_churn,
             x="percentage",
             y="PreferedOrderCat",
             color="Customer_Status",
             orientation='h',
             barmode="stack",
             category_orders={"PreferedOrderCat": category_order},
             text=cat_churn['percentage'].apply(lambda x: f'{x:.1f}%'),
             title="Category Risk Profile: Which Product Preferences Drive Attrition?",
             labels={'percentage': 'Percentage (%)', 'PreferedOrderCat': 'Preferred Category'},
             color_discrete_map={'Stay': '#bdc3c7', 'Churn': '#e74c3c'},
             template="plotly_white",
             height=500)

fig.update_traces(textposition='inside', insidetextanchor='middle')
fig.update_layout(xaxis=dict(showgrid=False, zeroline=False, showticklabels=False), xaxis_title=None)
fig.show()

Across several product categories, mobile phone segment shows the highest attrition rate at 27.4%, higher that onther category. The grocery here shows the most stable one with churn rate only 4.9%. This indicates that while the platform successfuly build loyalt that based on habit through daily need items, it still struggle to retain buyers from tech sectors which more price sensitive and prone to switching platform for better deals. This means a need for category focused retention strategies to secure the customer loyaty.

## **Preprocessing and Feature Engineering**

In [111]:
df_model = df_cleaned.drop(columns=['CustomerID'])
df_model = pd.get_dummies(df_model, columns=[
    'PreferredLoginDevice', 'PreferredPaymentMode',
    'Gender', 'PreferedOrderCat', 'MaritalStatus'
], drop_first=True)

df_model.head()

,Churn,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,...,PreferredPaymentMode_Debit Card,PreferredPaymentMode_E wallet,PreferredPaymentMode_UPI,Gender_Male,PreferedOrderCat_Grocery,PreferedOrderCat_Laptop & Accessory,PreferedOrderCat_Mobile Phone,PreferedOrderCat_Others,MaritalStatus_Married,MaritalStatus_Single
0,1,4,3,6,3,3,2,9,1,11.0,...,True,False,False,False,False,True,False,False,False,True
1,1,9,1,8,3,4,3,7,1,15.0,...,False,False,True,True,False,False,True,False,False,True
2,1,9,1,30,2,4,3,6,1,14.0,...,True,False,False,True,False,False,True,False,False,True
3,1,0,3,15,2,4,5,8,0,23.0,...,True,False,False,True,False,True,False,False,False,True
4,1,0,1,12,3,3,5,3,0,11.0,...,False,False,False,True,False,False,True,False,False,True


In [112]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training: {len(X_train)}")
print(f"Testing: {len(X_test)}")

Training: 4504
Testing: 1126


In [113]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [114]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE distribution: \n{y_train.value_counts()}")
print(f"After SMOTE distribution: \n{y_train_balanced.value_counts()}")

Before SMOTE distribution: 
Churn
0    3746
1     758
Name: count, dtype: int64
After SMOTE distribution: 
Churn
0    3746
1    3746
Name: count, dtype: int64


## **Modeling**

In [115]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [116]:
logreg = LogisticRegression(max_iter=5000, random_state=42)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

models = {'Logistic Regression': logreg, 'XGBoost': xgb, 'KNN': knn}
results = {}

for name, model in models.items():
    model.fit(X_train_balanced, y_train_balanced)
    y_pred = model.predict(X_test_scaled)

    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Recall (Churn)': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    }

df_results = pd.DataFrame(results).T
display(df_results)

,Accuracy,Recall (Churn),F1-Score
Logistic Regression,0.794849,0.821053,0.574586
XGBoost,0.952043,0.836842,0.854839
KNN,0.912078,0.973684,0.788913


Based on the initial evaluation, XGBoost was selected as the primary base model as it delivered to most optimal balance. This model achieved the highest accuracy 0.952 and F1-score 0.854. While KNN here yielded a higher recall score, its overall precision still lacking.

From this case, XGBoost is pretty effecitve at mapping non-linear relationship. This allowed the model to accurately interpret complex patterns discovered during EDA. Also, XGBoost is good at evaluating how multiple variables interact simultaneously.

In [117]:
def plot_plotly_cm(models, X_test, y_test):
    labels = ['Stay', 'Churn']

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f"CM: {name}" for name in models.keys()],
        horizontal_spacing=0.1
    )

    for i, (name, model) in enumerate(models.items()):
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)

        heatmap = go.Heatmap(
            z=cm[::-1],
            x=[f'Predicted {labels[0]}', f'Predicted {labels[1]}'],
            y=[f'Actual {labels[1]}', f'Actual {labels[0]}'],
            colorscale='Blues',
            showscale=False,
            text=cm[::-1],
            texttemplate="%{text}",
            textfont={"size": 16},
            hoverinfo="z"
        )

        fig.add_trace(heatmap, row=1, col=i+1)

    fig.update_layout(
        title_text="<b>Model Comparison: Customer Churn Confusion Matrix</b>",
        title_x=0.5,
        width=1200,
        height=550,
        template="plotly_white"
    )

    fig.update_xaxes(title_text="Predicted Status")
    fig.update_yaxes(title_text="Actual Status")

    fig.show()

In [118]:
plot_plotly_cm(models, X_test_scaled, y_test)

The confusion matrices reveals the business implications of each algorithm's predictions. From a cost-efficiency perspective, the baseline XGBoost offered the best operational balance, generating only 23 False Positives (FP). This minimizes the risk of the business wasting marketing budget on users who are already loyal.

KNN cast the widest net, achieving the highest sensitivity by correctly identifying 185 churners (TP) and missing only 5. However, this came at the steep cost of 94 FPs, which would lead to inflated operational costs in retention campaigns

Logistic Regression performed worst here, struggling with both high FN (34) and a massive 197 FP proving its inability to handle the dataset's non-linear complexity



In [119]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2]
}

xgb_tuned = XGBClassifier(random_state=42, eval_metric='logloss')

xgb_search = RandomizedSearchCV(xgb_tuned, param_distributions=param_grid,
                                n_iter=10, scoring='recall', cv=5, random_state=42, n_jobs=-1)

xgb_search.fit(X_train_balanced, y_train_balanced)

print(f"Best Parameters: {xgb_search.best_params_}")
best_xgb = xgb_search.best_estimator_

Best Parameters: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 0.7}


In [120]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

y_pred_tuned = best_xgb.predict(X_test_scaled)

print("--- Tuned XGBoost Performance ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_tuned):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_tuned):.4f}")

--- Tuned XGBoost Performance ---
Accuracy : 0.9849
Recall   : 0.9526
F1-Score : 0.9551


In [121]:
compare_models = {
    'Baseline XGBoost': xgb,
    'Tuned XGBoost': best_xgb
}

plot_plotly_cm(compare_models, X_test_scaled, y_test)

In [122]:
df_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_xgb.feature_importances_
}).sort_values(by='Importance', ascending=True)

fig = px.bar(df_importance,
             x='Importance',
             y='Feature',
             orientation='h',
             title='Feature Importance: What Drives Churn According to XGBoost?</b>',
             labels={'Importance': 'Importance Score', 'Feature': 'Features'},
             template='plotly_white',
             color='Importance',
             color_continuous_scale='Tealgrn')

fig.update_layout(
    height=800,
    yaxis={'categoryorder':'total ascending'},
    title_x=0.5,
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.show()

From the optimized XGBoost model, the result above show the feature importance scores.
- Tenure is the most influential predictor of churn, it means that the longer a customer stays with the paltform, the more resilitent they become to attrition.
- Complain ranks as the second strongest predictor, serving as an immediate red flag and a direct trigger for churn if not handled properly by customer service.
- NumberOfDeviceRegistered and specific tech categories ('Laptop & Accessory' and 'Mobile Phone') rank highly. This suggests a high-risk segment of price-sensitive customers who register multiple devices to exploit one-off discounts for expensive electronics, only to churn immediately after

# **Business Impact Simulation**

In [123]:
promo_cost = 50000
customer_value = 300000
retention_rate = 0.50

TP = 181
FP = 8
FN = 9
TN = 928

total_customers = TP + FP + FN + TN
cost_without_model = total_customers * promo_cost
revenue_saved_without_model = (TP + FN) * retention_rate * customer_value
net_profit_without_model = revenue_saved_without_model - cost_without_model

cost_with_model = (TP + FP) * promo_cost
revenue_saved_with_model = (TP * retention_rate) * customer_value
net_profit_with_model = revenue_saved_with_model - cost_with_model

categories = ['Without Model', 'With Tuned XGBoost']
costs = [cost_without_model, cost_with_model]
profits = [net_profit_without_model, net_profit_with_model]

fig = go.Figure(data=[
    go.Bar(name='Marketing Cost',
           x=categories,
           y=costs,
           marker_color='#e74c3c',
           text=[f"Rp {val/1e6:.1f}M" for val in costs],
           textposition='auto'),

    go.Bar(name='Net Profit / (Loss)',
           x=categories,
           y=profits,
           marker_color=['#c0392b' if val < 0 else '#2ecc71' for val in profits],
           text=[f"Rp {val/1e6:.1f}M" for val in profits],
           textposition='auto')
])

fig.update_layout(
    barmode='group',
    title='Financial Impact',
    title_x=0.5,
    yaxis_title='Amount (IDR)',
    template='plotly_white',
    width=900,
    height=600,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)')
)

fig.add_hline(y=0, line_width=2, line_color="black")

fig.show()

summary_df = pd.DataFrame({
    'Strategy': categories,
    'Targeted Audience': [total_customers, (TP+FP)],
    'Total Cost (Rp)': [f"{cost_without_model:,.0f}", f"{cost_with_model:,.0f}"],
    'Net Profit (Rp)': [f"{net_profit_without_model:,.0f}", f"{net_profit_with_model:,.0f}"]
})
display(summary_df)

,Strategy,Targeted Audience,Total Cost (Rp),Net Profit (Rp)
0,Without Model,1126,"56,300,000","-27,800,000"
1,With Tuned XGBoost,189,"9,450,000","17,700,000"
